In [1]:
!pip install pymysql sqlalchemy --quiet

In [2]:
import pandas as pd, numpy as np, boto3, json, pickle, warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model    import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.metrics         import accuracy_score, f1_score, roc_auc_score
from sqlalchemy              import create_engine
import sagemaker

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [3]:
sm_client = boto3.client('secretsmanager')
secret = json.loads(
    sm_client.get_secret_value(SecretId='metrocity/rds/credentials')['SecretString']
)
print("Credentials retrieved from Secrets Manager.")

Credentials retrieved from Secrets Manager.


In [5]:
engine = create_engine(
    f"mysql+pymysql://{secret['username']}:{secret['password']}"
    f"@{secret['host']}/{secret['dbname']}"
)

ml_df = pd.read_sql("""
    SELECT  a.accident_id,
            l.location_name,
            w.weather_condition,
            a.road_condition,
            v.vehicle_type,
            d.hour,
            d.is_peak_hour,
            d.is_weekend,
            d.month,
            d.quarter,
            a.number_of_vehicles,
            a.traffic_density_num,
            a.adverse_weather,
            a.hazardous_road,
            a.is_fatal_or_severe
    FROM    fact_accident a
    JOIN    dim_location l ON a.location_id = l.location_id
    JOIN    dim_weather  w ON a.weather_id  = w.weather_id
    JOIN    dim_vehicle  v ON a.vehicle_id  = v.vehicle_id
    JOIN    dim_date     d ON a.date_id     = d.date_id
""", engine)

print(f"ML dataset shape: {ml_df.shape}")
print(f"Target split: {ml_df.is_fatal_or_severe.value_counts().to_dict()}")

ML dataset shape: (5000, 15)
Target split: {0: 2510, 1: 2490}


In [6]:
FEATURES = ['location_name','weather_condition','road_condition','vehicle_type',
            'hour','is_peak_hour','is_weekend','month','quarter',
            'number_of_vehicles','traffic_density_num','adverse_weather','hazardous_road']
TARGET   = 'is_fatal_or_severe'

# Encode categorical columns with LabelEncoder
cat_cols = ["location_name","weather_condition","road_condition","vehicle_type"]
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    ml_df[col] = le.fit_transform(ml_df[col])
    encoders[col] = le

X = ml_df[FEATURES]
y = ml_df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     stratify=y, random_state=42)

In [7]:
scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_ts_sc = scaler.transform(X_test)
cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

MODELS = {
    'Random Forest'      : (RandomForestClassifier(n_estimators=150, max_depth=10,
                                                    random_state=42, n_jobs=-1), False),
    'Gradient Boosting'  : (GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                                        max_depth=5, random_state=42), False),
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), True),
}

results = {}
for name, (model, needs_scaling) in MODELS.items():
    Xtr = X_tr_sc if needs_scaling else X_train
    Xts = X_ts_sc if needs_scaling else X_test
    model.fit(Xtr, y_train)
    preds = model.predict(Xts)
    proba = model.predict_proba(Xts)[:,1]
    cv_scores = cross_val_score(model, Xtr, y_train, cv=cv, scoring="f1", n_jobs=-1)
    results[name] = {
        'model':       model,
        'needs_scale': needs_scaling,
        'accuracy':    accuracy_score(y_test, preds),
        'f1':          f1_score(y_test, preds),
        'roc_auc':     roc_auc_score(y_test, proba),
        'cv_f1_mean':  cv_scores.mean(),
        'cv_f1_std':   cv_scores.std(),
    }
    print(f"{name}: Acc={results[name]["accuracy"]:.3f}  F1={results[name]["f1"]:.3f}  AUC={results[name]["roc_auc"]:.3f}")

best_name  = max(results, key=lambda k: results[k]['roc_auc'])
best       = results[best_name]
best_model = best['model']
best_scaled= best['needs_scale']
print(f"\nBest model: {best_name}")

Random Forest: Acc=0.536  F1=0.507  AUC=0.553
Gradient Boosting: Acc=0.524  F1=0.506  AUC=0.515
Logistic Regression: Acc=0.516  F1=0.506  AUC=0.527

Best model: Random Forest


In [8]:
from sagemaker.experiments.run import Run
 
with Run(experiment_name='metrocity-severity-prediction',
         run_name=f'run-{best_name.lower().replace(" ","-")}') as run:
    run.log_parameter('model_type',   best_name)
    run.log_parameter('n_features',   len(FEATURES))
    run.log_parameter('train_size',   len(X_train))
    run.log_metric('test_accuracy',   float(best['accuracy']))
    run.log_metric('test_f1',         float(best['f1']))
    run.log_metric('roc_auc',         float(best['roc_auc']))
    run.log_metric('cv_f1_mean',      float(best['cv_f1_mean']))
 
print("Experiment logged to SageMaker Experiments.")


Experiment logged to SageMaker Experiments.


In [9]:
X_full   = scaler.transform(X) if best_scaled else X
pred_prob = best_model.predict_proba(X_full)[:,1].round(4)
pred_cls  = best_model.predict(X_full)

def risk_tier(p):
    if p < 0.35:   return "Low Risk"
    if p < 0.50:   return "Moderate Risk"
    if p < 0.70:   return "High Risk"
    return "Critical Risk"

risk_df = pd.DataFrame({
    'accident_id':         ml_df['accident_id'],
    'location_name':       ml_df['location_name'].map(
        dict(enumerate(encoders['location_name'].classes_))),
    'predicted_risk_prob': pred_prob,
    'predicted_class':     pred_cls,
    'actual_fatal_severe': ml_df['is_fatal_or_severe'],
    'risk_tier':           [risk_tier(p) for p in pred_prob]
})
risk_df.to_sql("ml_accident_risk", engine, if_exists="replace", index=False)
print(f"ml_accident_risk loaded: {len(risk_df)} rows.")
print(risk_df.risk_tier.value_counts())

ml_accident_risk loaded: 5000 rows.
risk_tier
High Risk        2054
Moderate Risk    1971
Low Risk          651
Critical Risk     324
Name: count, dtype: int64


In [10]:
BUCKET = "metrocity-harshita-data"
model_bytes = pickle.dumps(best_model)
boto3.client('s3').put_object(
    Bucket=BUCKET,
    Key=f"ml-artifacts/{best_name.replace(" ","_")}_model.pkl",
    Body=model_bytes
)
print(f"Model artifact saved to s3://{BUCKET}/ml-artifacts/")

Model artifact saved to s3://metrocity-harshita-data/ml-artifacts/
